실행 위치: Google Colab

# 1. vllm 라이브러리 설치

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
!uv pip install --system vllm==0.11.0 torch==2.8.0 transformers==4.57.6

# 2. 임베딩 모델 로드

In [ ]:
!vllm serve dragonkue/BGE-m3-ko --gpu-memory-utilization 0.4 --port 8000 > vllm_server.log 2>&1 &

In [ ]:
!cat vllm_server.log

## 임베딩 모델 추론

In [ ]:
payload = {
    "model": "dragonkue/BGE-m3-ko",
    "input": [
        "서울의 경복궁은 어떤 곳이야?",
        "서울의 경복궁과 부산의 해운대 중 어떤 곳을 더 추천해?",
        "트랜스포머의 어텐션 원리에 대해 설명해주세요."
    ]
}

headers={"Content-Type": "application/json"}

In [ ]:
import httpx, json


async with httpx.AsyncClient(timeout=30) as client:
    response = await client.post(
        "http://localhost:8000/v1/embeddings",
        json=payload,
        headers=headers
    )
    response.raise_for_status()
    data = response.json()
    print(json.dumps(data, indent=2, ensure_ascii=False))


- 임베딩된 결과 벡터의 차원 수 확인

In [ ]:
vector1 = data['data'][0]['embedding']
vector2 = data['data'][1]['embedding']
vector3 = data['data'][2]['embedding']

print(len(vector1))
print(len(vector2))
print(len(vector3))

## 임베딩 벡터간 유사도 계산 (코사인 유사도)

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    v1 = np.array(vec1, dtype=np.float32)
    v2 = np.array(vec2, dtype=np.float32)

    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [ ]:
print(cosine_similarity(vector1, vector2))
print(cosine_similarity(vector2, vector3))
print(cosine_similarity(vector1, vector3))

# 3. 리랭커 모델 로드

In [ ]:
!vllm serve BAAI/bge-reranker-v2-m3 --gpu-memory-utilization=0.4 --port 8001 > vllm_server_reranker.log 2>&1 &

In [ ]:
!cat vllm_server_reranker.log

## 리랭커 모델 추론

In [ ]:
payload = {
    "model": "BAAI/bge-reranker-v2-m3",
    "query": "공정 결함의 원인이 뭐야?",
    "documents": [
        "결함은 주로 온도 조절 실패에서 발생합니다.",
        "오늘 점심 메뉴는 돈가스입니다.",
        "센서 데이터 수집 오류가 결함의 주원인입니다.",
        "각 패널은 공정을 거쳐 순서대로 진행됩니다.",
        "서울의 경복궁은 외국인들이 가장 많이 찾는 곳 중 하나입니다."
    ],
    "top_n": 2
}

headers={"Content-Type": "application/json"}

- /rerank: vLLM에서 자체 개발한 엔드포인트 (미추천)
- /v1/rerank: OpenAI 스타일
- /v2/rerank: Cohere 스타일

In [ ]:
import httpx, json


async with httpx.AsyncClient(timeout=30) as client:
    response = await client.post(
        "http://localhost:8001/v1/rerank",
        #http://localhost:8001/v2/rerank",
        json=payload,
        headers=headers
    )
    response.raise_for_status()
    data = response.json()
    print(json.dumps(data, indent=2, ensure_ascii=False))

# 4. GPU 메모리 사용량 확인

In [ ]:
!nvidia-smi

- 언어 모델을 로드했을 때와 달리 메모리 여유 공간을 KV-Cache가 점유하지 않으며
- 메모리에 여유가 많은 것을 알 수 있음